In [1]:
import pandas as pd
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

base = Path.cwd()
data_path = base / "Data" / "Raw-AI-Dataset.csv"

if not data_path.exists():
    data_path = (base / ".." / "Data" / "Raw-AI-Dataset.csv").resolve()

df = pd.read_csv(data_path)
df.head()

,Age,Education_Level,Years_of_Experience,Industry,Job_Role,Company_Size,Job_Level,Routine_Task_Percentage,Creativity_Requirement,Human_Interaction_Level,AI_Adoption_Level,Number_of_AI_Tools_Used,AI_Usage_Hours_Per_Week,Tasks_Automated_Percentage,AI_Training_Hours,Layoff_Risk
0,59,Master's,6,Finance,Accountant,Medium,Entry,84,21,94,Medium,5,7,40,26,High
1,44,Master's,14,Manufacturing,Production Supervisor,Small,Entry,30,84,68,Low,2,2,14,9,Low
2,36,Bachelor's,7,Retail,Store Manager,Medium,Senior,12,86,71,Low,0,1,6,3,Low
3,27,Bachelor's,6,Finance,Auditor,Large,Entry,90,18,21,Medium,3,10,67,26,High
4,49,High School,12,Finance,Auditor,Small,Entry,49,52,72,Medium,5,13,26,19,Medium


In [2]:
# 1. Basic Cleaning
# Drop duplicate rows to ensure clean modeling data
df = df.drop_duplicates()

# 2. Target Encoding
# Map the ordinal target to numerical values for classification
risk_mapping = {"Low": 0, "Medium": 1, "High": 2}
if df["Layoff_Risk"].dtype == "object":
    df["Layoff_Risk"] = df["Layoff_Risk"].map(risk_mapping)

# 3. Categorical Encoding
# Convert 'Industry' to binary indicators. drop_first=True prevents the dummy variable trap.
df = pd.get_dummies(df, columns=["Industry"], drop_first=True, dtype=int)

# 4. Feature Engineering
# Create interaction terms aligned with your research focus
if "Routine_Task_Percentage" in df.columns and "Creativity_Requirement" in df.columns:
    df["Automation_Ratio"] = df["Routine_Task_Percentage"] / (df["Creativity_Requirement"] + 1)

if "Human_Interaction_Level" in df.columns and "Years_of_Experience" in df.columns:
    df["Shield_Index"] = df["Human_Interaction_Level"] + df["Years_of_Experience"]

# 5. Multicollinearity Handling
# Drop variables known to have high Variance Inflation Factor (VIF > 5.0)
if "AI_Usage_Hours_Per_Week" in df.columns:
    df = df.drop(columns=["AI_Usage_Hours_Per_Week"])

# 6. Stratified 3-Way Split (70% Train, 15% Validation, 15% Test)
X = df.drop(columns=["Layoff_Risk"])
y = df["Layoff_Risk"]

# Initial split: 70% Train, 30% Temp (Stratify maintains class balance)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Second split: Divide the 30% Temp evenly into Validation and Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# 7. Data Leakage Prevention (Scaling)
# Crucial: Fit the StandardScaler ONLY on the training data.
scaler = StandardScaler()
scale_cols = ["Years_of_Experience", "Age", "Automation_Ratio", "Shield_Index"]
scale_cols = [c for c in scale_cols if c in X_train.columns] # Defensive check

X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_val[scale_cols] = scaler.transform(X_val[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

print(f"Data cleaned and split! Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

Data cleaned and split! Train: 14000, Val: 3000, Test: 3000


In [3]:
display(X_train.head())

,Age,Education_Level,Years_of_Experience,Job_Role,Company_Size,Job_Level,Routine_Task_Percentage,Creativity_Requirement,Human_Interaction_Level,AI_Adoption_Level,...,AI_Training_Hours,Industry_Finance,Industry_Healthcare,Industry_IT,Industry_Logistics,Industry_Manufacturing,Industry_Retail,Industry_Telecom,Automation_Ratio,Shield_Index
16459,-0.382387,Master's,0.363657,Data Analyst,Small,Senior,46,61,54,Low,...,7,0,0,1,0,0,0,0,-0.266551,-0.196429
9650,0.833292,Bachelor's,0.786321,Accountant,Small,Entry,62,46,52,Medium,...,11,1,0,0,0,0,0,0,-0.226388,-0.196429
12495,1.180628,Bachelor's,-0.693002,Production Supervisor,Medium,Mid,22,50,79,Low,...,1,0,0,0,0,1,0,0,-0.288160,0.678687
1967,0.572789,Bachelor's,-1.115666,Academic Coordinator,Medium,Entry,21,70,71,Low,...,8,0,0,0,0,0,0,0,-0.297595,0.241129
11571,0.225452,Bachelor's,0.786321,Warehouse Manager,Small,Entry,19,84,62,Low,...,2,0,0,0,1,0,0,0,-0.302622,0.241129


In [4]:
print("--- Features (Predictors) ---")
print("X_train shape:", X_train.shape)
print("X_val shape:  ", X_val.shape)
print("X_test shape: ", X_test.shape)

print("\n--- Targets (Layoff_Risk) ---")
print("y_train shape:", y_train.shape)
print("y_val shape:  ", y_val.shape)
print("y_test shape: ", y_test.shape)

--- Features (Predictors) ---
X_train shape: (14000, 22)
X_val shape:   (3000, 22)
X_test shape:  (3000, 22)

--- Targets (Layoff_Risk) ---
y_train shape: (14000,)
y_val shape:   (3000,)
y_test shape:  (3000,)


In [5]:
from pathlib import Path

# Define your exact target directory using a raw string (r"...") to handle Windows backslashes
target_dir = Path(r"C:\Users\19496\OneDrive\Desktop\MSAAI\Final Project\AAI500-Final-Project\Data")

# Ensure the folder exists
target_dir.mkdir(parents=True, exist_ok=True)

# Save feature sets (predictors)
X_train.to_csv(target_dir / "processed_X_train.csv", index=False)
X_val.to_csv(target_dir / "processed_X_val.csv", index=False)
X_test.to_csv(target_dir / "processed_X_test.csv", index=False)

# Save target sets (Layoff_Risk)
y_train.to_csv(target_dir / "processed_y_train.csv", index=False)
y_val.to_csv(target_dir / "processed_y_val.csv", index=False)
y_test.to_csv(target_dir / "processed_y_test.csv", index=False)

print(f"All 6 datasets successfully saved to:\n{target_dir}")

All 6 datasets successfully saved to:
C:\Users\19496\OneDrive\Desktop\MSAAI\Final Project\AAI500-Final-Project\Data
